# Homework: Breaking the Sequence Model
## The Challenge:
In class, we built an LSTM to classify movie reviews, and I made two big theoretical claims:

1. Standard RNNs have amnesia. They suffer from the Vanishing Gradient problem and forget early words.
2. We shouldn't delete stop words. Unlike old Machine Learning, Deep Learning needs words like "not" to understand context.

Your homework is to experimentally prove (or disprove!) both of these claims by modifying the code we wrote in class.

## Task 1: The Vanilla RNN Showdown
PyTorch makes it incredibly easy to swap out neural network architectures.
Below is the SentimentLSTM we built in class. Your task is to write a new class called SentimentRNN that uses PyTorch's standard nn.RNN module instead of nn.LSTM.

Hint: The standard nn.RNN only outputs the hidden state, it does not have a cell state (the conveyor belt). You will need to adjust the forward function so it doesn't crash trying to unpack a cell state that doesn't exist!

In [ ]:
import torch
import torch.nn as nn

# --- The Model from Class ---
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        embedded = self.embedding(text)
        lstm_out, (hidden, cell) = self.lstm(embedded) # LSTM has Hidden AND Cell
        final_hidden_state = hidden[-1]
        out = self.fc(final_hidden_state)
        return self.sigmoid(out).squeeze()

# --- YOUR TURN: Build the Vanilla RNN ---
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # TODO: Initialize a standard RNN here instead of LSTM
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        embedded = self.embedding(text)

        # TODO: Pass the embedded text through the RNN.
        # CAREFUL: nn.RNN returns (output, hidden). There is no 'cell' state!

        rnn_out, hidden = self.rnn(embedded)
        final_hidden_state = hidden[-1]
        out = self.fc(final_hidden_state)
        return self.sigmoid(out).squeeze()

### Thought Experiment (Answer in a comment block below):
If you train both of these models on the IMDB dataset for 5 epochs, the SentimentLSTM will likely reach ~85% accuracy, while the SentimentRNN might struggle to pass 60%. Based on the lecture, explain mathematically why the vanilla RNN is failing to classify a 200-word movie review accurately.

The reason the Vanilla RNN struggles with a 200-word review while the LSTM succeeds comes down to the Vanishing Gradient Problem.Mathematically, during backpropagation through time (BPTT), we calculate the gradient of the loss with respect to the weights. This involves a chain of matrix multiplications of the recurrent weight $W_{hh}$. For a sequence of length $T=200$:$$\frac{\partial h_t}{\partial h_1} = \prod_{k=2}^{t} \frac{\partial h_k}{\partial h_{k-1}}$$In a Vanilla RNN, the derivative involves the activation function (like $tanh$). If the weights are small, the gradient shrinks exponentially as it moves back through 200 steps ($0.9^{200} \approx 0$). By the time the "signal" from the first word of the movie review reaches the weights at the end, it has effectively vanished. The model "forgets" the beginning of the review.The LSTM solves this with the Constant Error Carousel. Its "Cell State" ($c_t$) allows information to flow through a linear path with only minor additive changes (via the forget and input gates), meaning the gradient can stay close to 1.0 even over 200 steps.

## Task 2: The "Old Rules" Experiment
In class, I told you that deleting "Stop Words" (the, is, not, at) is a bad idea for Deep Learning sequence models. Let's see why.

Write a basic Python function that takes a movie review and removes the stop words. Then, look at the output and explain why feeding this "cleaned" text to an LSTM might cause it to make the wrong prediction.

In [ ]:
# A small list of common stop words to remove
stop_words = ["i", "me", "my", "the", "and", "is", "in", "it", "not", "was", "a", "an", "to"]


def remove_stop_words(review):
    stop_words = {"the", "is", "not", "at", "a", "and", "it"} # Example list
    words = review.lower().split()
    cleaned_review = [w for w in words if w not in stop_words]
    return " ".join(cleaned_review)


# Test your function on this specific review:
review = "i thought the movie was not bad and it was a joy to watch"
cleaned_review = remove_stop_words(review)

print(f"Original: {review}")
print(f"Cleaned:  {cleaned_review}")

Original: i thought the movie was not bad and it was a joy to watch
Cleaned:  i thought movie was bad was joy to watch


### Analysis (Answer in a comment block below):
Look at the cleaned_review you just generated. If you pass this cleaned string into our SentimentLSTM, what do you think it will predict (Positive or Negative)? Why is this a problem?

In [ ]:
vocab = {"i": 1, "thought": 2, "the": 3, "movie": 4, "was": 5, "not": 6, "bad": 7, "and": 8, "it": 9, "a": 10, "joy": 11, "to": 12, "watch": 13}
stop_words = {"the", "is", "not", "at", "a", "and", "it"}

def text_to_tensor(text, vocab):
    indices = [vocab[w] for w in text.lower().split() if w in vocab]
    return torch.LongTensor([indices])
model = SentimentLSTM(len(vocab) + 1, embed_dim=8, hidden_dim=16)
model.eval()

review = "i thought the movie was not bad and it was a joy to watch"
cleaned_review = remove_stop_words(review)

original_tensor = text_to_tensor(review, vocab)
cleaned_tensor = text_to_tensor(cleaned_review, vocab)

with torch.no_grad():
    original_pred = model(original_tensor).item()
    cleaned_pred = model(cleaned_tensor).item()

print(f"Original Review: '{review}'")
print(f"Cleaned Review:  '{cleaned_review}'")
print("-" * 30)
print(f"LSTM Prediction (Original): {original_pred:.4f}")
print(f"LSTM Prediction (Cleaned):  {cleaned_pred:.4f}")

Original Review: 'i thought the movie was not bad and it was a joy to watch'
Cleaned Review:  'i thought movie was bad was joy to watch'
------------------------------
LSTM Prediction (Original): 0.5852
LSTM Prediction (Cleaned):  0.5793


While my experimental output above gave random predictions (~0.58) because the model is currently untrained, if we passed this into a FULLY TRAINED SentimentLSTM, it would likely predict the cleaned text as NEGATIVE.

Why this is a problem:
1. Negation Flip: The original review is positive ("not bad", "joy to watch"). By removing the stop word "not", the phrase becomes "movie was bad".
2. Loss of Context: Deep Learning sequence models rely on the spatial relationship between words. Removing the "glue" words (is, the, it) destroys the natural linguistic structure the LSTM was trained to process.
Therefore, applying traditional stop-word removal to sequence models often destroys critical context and flips the intended sentiment.